# Refinitiv Finalize-Only Helper (Colab)

This notebook is intended for the post-staging workflow:

- fetch or stage API outputs on the local machine
- copy the run root to Google Drive
- run individual `finalize_only` stages in Colab
- run the lightweight non-API stages in `full` mode between them when needed

It assumes the repository is available in Colab and the required project dependencies are already installed.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive', force_remount=False)


def _resolve_drive_root() -> Path:
    for candidate in (
        Path('/content/drive/MyDrive'),
        Path('/content/drive/My Drive'),
        Path('/content/drive'),
    ):
        if candidate.exists():
            return candidate
    return Path('/content/drive')


def _resolve_repo_root(drive_root: Path) -> Path:
    override = os.environ.get('NLP_THESIS_REPO_ROOT', '').strip()
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend(
        [
            Path('/content/NLP_Thesis'),
            drive_root / 'Data_LM' / 'code' / 'NLP_Thesis',
            drive_root / 'NLP_Thesis',
        ]
    )
    for candidate in candidates:
        if (candidate / 'src' / 'thesis_pkg' / 'pipeline.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate the repo root. Set NLP_THESIS_REPO_ROOT or place the repo in a standard Colab path.'
    )


DRIVE_ROOT = _resolve_drive_root()
REPO_ROOT = _resolve_repo_root(DRIVE_ROOT)
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DRIVE_WORK_ROOT = DRIVE_ROOT / 'Data_LM'
default_run_root = DRIVE_WORK_ROOT / 'results' / 'sec_ccm_unified_runner'
fallback_run_root = DRIVE_WORK_ROOT / 'full_data_run'
run_root_override = os.environ.get('REFINITIV_RUN_ROOT', '').strip()
RUN_ROOT = (
    Path(run_root_override).expanduser()
    if run_root_override
    else default_run_root
    if default_run_root.exists()
    else fallback_run_root
)

print({'DRIVE_ROOT': str(DRIVE_ROOT), 'REPO_ROOT': str(REPO_ROOT), 'RUN_ROOT': str(RUN_ROOT)})

In [ ]:
from thesis_pkg.notebooks_and_scripts import refinitiv_local_api_runner as ref_runner

BATCH_PROFILE = 'local_safe'


def run_stage(stage: str, *, api_stage_mode: str = 'full', batch_profile: str = BATCH_PROFILE, extra_args: list[str] | None = None) -> int:
    argv = [
        '--run-root',
        str(RUN_ROOT),
        '--stage-start',
        stage,
        '--stage-stop',
        stage,
        '--api-stage-mode',
        api_stage_mode,
        '--batch-profile',
        batch_profile,
    ]
    if extra_args:
        argv.extend(extra_args)
    print('Running:', ' '.join(argv))
    exit_code = ref_runner.main(argv)
    print('Exit code:', exit_code)
    return exit_code


def run_stage_list(stages: list[str], *, batch_profile: str = BATCH_PROFILE, extra_args: list[str] | None = None) -> int:
    argv = [
        '--run-root',
        str(RUN_ROOT),
        '--stage-list',
        ','.join(stages),
        '--api-stage-mode',
        'full',
        '--batch-profile',
        batch_profile,
    ]
    if extra_args:
        argv.extend(extra_args)
    print('Running:', ' '.join(argv))
    exit_code = ref_runner.main(argv)
    print('Exit code:', exit_code)
    return exit_code


def show_stage_paths(stage: str) -> None:
    paths = ref_runner._resolve_run_paths(Path(RUN_ROOT), reviewed_ticker_allowlist_path=None)
    print('inputs:', ref_runner._stage_input_paths(paths, stage))
    print('outputs:', ref_runner._stage_output_paths(paths, stage))
    print('resume sentinels:', ref_runner._stage_resume_sentinels(paths, stage))
    print('stage manifests:', ref_runner._stage_manifest_outputs(paths, stage))
    print('fetch manifests:', ref_runner._stage_fetch_manifest_outputs(paths, stage))


print('Stage order:', ref_runner.STAGE_ORDER)
print('API stages:', sorted(ref_runner.API_STAGES))

## Typical Colab Sequence

Use `finalize_only` only after the local machine has already written the stage ledger, request log, and staging parquet files into the copied `RUN_ROOT` on Drive.

Typical order when ownership was fetched locally first:

1. `ownership_api` in `finalize_only`
2. `authority` in `full`
3. `analyst_request_groups` in `full`
4. `analyst_actuals_api` in `finalize_only` after local staging exists
5. `analyst_estimates_api` in `finalize_only` after local staging exists
6. `analyst_normalize` in `full`
7. `doc_exact_api` in `finalize_only` after local staging exists
8. `doc_fallback_api` in `finalize_only` after local staging exists
9. `doc_finalize` in `full`


In [ ]:
# Inspect one stage before running it.
# show_stage_paths('ownership_api')

# Ownership finalize after local staging.
# run_stage('ownership_api', api_stage_mode='finalize_only')

# Downstream non-API stages in full mode.
# run_stage('authority')
# run_stage('analyst_request_groups')

# Analyst finalize steps after local fetch/staging.
# run_stage('analyst_actuals_api', api_stage_mode='finalize_only')
# run_stage('analyst_estimates_api', api_stage_mode='finalize_only')
# run_stage('analyst_normalize')

# Document ownership finalize steps after local fetch/staging.
# run_stage('doc_exact_api', api_stage_mode='finalize_only')
# run_stage('doc_fallback_api', api_stage_mode='finalize_only')
# run_stage('doc_finalize')

# Optional: run the full local-LSEG block in one Colab step instead of stage-by-stage.
# run_stage_list([
#     'authority',
#     'analyst_request_groups',
#     'analyst_actuals_api',
#     'analyst_estimates_api',
#     'doc_exact_api',
#     'doc_fallback_api',
# ])